In [10]:
import pandas as pd
import networkx as nx
import numpy as np 

In [6]:
# Initializing the graph from data

nodes = pd.read_csv(
    "../data/hep-th-slacdates.tar.gz", 
    sep=r'\s+', 
    header=None, 
    names=["id", "date"], 
    dtype=str  
)


edges = pd.read_csv(
    "../data/hep-th-citations.tar.gz", 
    sep=r'\s+', 
    header=None, 
    names=["From", "To"], 
    dtype=str
)

# 3. Get unique node IDs from both columns of the edge list
node_ids = pd.unique(edges[["From", "To"]].values.ravel())

# 4. Get actual nodes listed in edge list
nodes_used = nodes[nodes["id"].isin(node_ids)].copy()

# 5. Make a directed graph smoothly
g_directed = nx.DiGraph()

# Convert the filtered nodes dataframe into a dictionary of attributes and load them
node_attributes = nodes_used.set_index("id").to_dict("index")
g_directed.add_nodes_from(node_attributes.keys())
nx.set_node_attributes(g_directed, node_attributes)

# Add the edges directly from the edge dataframe tuples
g_directed.add_edges_from(edges.itertuples(index=False, name=None))

# Print properties to verify success
print("--- GRAPH INITIALIZATION SUCCESSFUL ---")
print(f"Nodes in graph: {g_directed.number_of_nodes()}")
print(f"Edges in graph: {g_directed.number_of_edges()}")

--- GRAPH INITIALIZATION SUCCESSFUL ---
Nodes in graph: 27770
Edges in graph: 352807


In [7]:
# Null values check
print(f"Nulls in From column: {edges['From'].isna().sum()}")
print(f"Nulls in To column:   {edges['To'].isna().sum()}")

Nulls in From column: 0
Nulls in To column:   0


In [8]:
# Identify cross-listed papers (starting with "11")
crosslisted_dates = nodes[nodes["id"].str.startswith("11", na=False)].shape[0]
crosslisted_citations = edges[edges["From"].str.startswith("11", na=False) | edges["To"].str.startswith("11", na=False)].shape[0]
print(f"Crosslisted rows in dates file: {crosslisted_dates}")
print(f"Crosslisted rows in citations file: {crosslisted_citations}")

Crosslisted rows in dates file: 0
Crosslisted rows in citations file: 0


In [9]:
# Get all unique nodes active in edge pairs
edges_nodes = pd.unique(edges[["From", "To"]].values.ravel())
vertices_nodes = nodes["id"].unique()

In [11]:
# Find structural mismatch anomalies
problem_nodes = np.setdiff1d(edges_nodes, vertices_nodes)
print(f"\n--- TARGETED GRAPH MATCHING DIAGNOSTIC ---")
print(f"Total unique nodes found in edge links: {len(edges_nodes)}")
print(f"Total unique nodes found in dates file: {len(vertices_nodes)}")
print(f"Number of offending nodes missing metadata: {len(problem_nodes)}")


--- TARGETED GRAPH MATCHING DIAGNOSTIC ---
Total unique nodes found in edge links: 27770
Total unique nodes found in dates file: 29555
Number of offending nodes missing metadata: 0


Summary of the network properties

In [ ]:
in_deg = dict(g_directed.in_degree())
out_deg = dict(g_directed.out_degree())
in_vals, out_vals = list(in_deg.values()), list(out_deg.values())
is_strongly_connected = nx.is_strongly_connected(g_directed)
is_weakly_connected = nx.is_weakly_connected(g_directed)

scc = list(nx.strongly_connected_components(g_directed))
wcc = list(nx.weakly_connected_components(g_directed))
clustering = list(nx.clustering(g_directed).values())
# avg_path_length = nx.average_shortest_path_length(g_directed)

summary_data = {
    "Network Property": [
        "Total Vertices (Papers)", "Total Edges (Citations)", "Network Density",
        "Strongly Connected ?", "Weakly Connected ?",
        "Mean In-Degree (Citations Received)", "Max In-Degree (Top Paper)",
        "Mean Out-Degree (References Given)", "Max Out-Degree (Longest Bibliography)",
        "Total Weakly Connected Components", "Size of Largest Connected Component (WCC)",
        "Total Strongly Connected Components (Cycles)", "Size of Largest Cyclic Component (SCC)",
        "Mean Local Clustering Coefficient"
    ],
    "Value": [
        g_directed.number_of_nodes(), g_directed.number_of_edges(), f"{nx.density(g_directed):.6f}",
        is_strongly_connected, is_weakly_connected,  
        f"{np.mean(in_vals):.2f}", np.max(in_vals),
        f"{np.mean(out_vals):.2f}", np.max(out_vals),
        len(wcc), max(len(c) for c in wcc),
        len(scc), max(len(c) for c in scc),
        f"{np.mean(clustering):.4f}",
        # f"{avg_path_length:.4f}"
    ]
}

network_summary = pd.DataFrame(summary_data)
print("\n=== THE TOPOLOGICAL PROFILE OF THE NETWORK ===")
print(network_summary.to_string(index=False))


=== THE TOPOLOGICAL PROFILE OF THE NETWORK ===
                            Network Property    Value
                     Total Vertices (Papers)    27770
                     Total Edges (Citations)   352807
                             Network Density 0.000458
                        Strongly Connected ?    False
                          Weakly Connected ?    False
         Mean In-Degree (Citations Received)    12.70
                   Max In-Degree (Top Paper)     2414
          Mean Out-Degree (References Given)    12.70
       Max Out-Degree (Longest Bibliography)      562
           Total Weakly Connected Components      143
   Size of Largest Connected Component (WCC)    27400
Total Strongly Connected Components (Cycles)    20086
      Size of Largest Cyclic Component (SCC)     7464
           Mean Local Clustering Coefficient   0.1569


In [15]:
largest_wcc_nodes = max(nx.weakly_connected_components(g_directed), key=len)
g_largest_wcc = g_directed.subgraph(largest_wcc_nodes)

g_undirected_wcc = g_largest_wcc.to_undirected()

avg_path_length = nx.average_shortest_path_length(g_undirected_wcc)

print(f"Average Path Length of the wCC {avg_path_length}")

Average Path Length of the wCC 4.278680083960654
